<a href="https://colab.research.google.com/github/braltoids0089/AGRO-BIOTECHNOLOGY/blob/main/LAYER3_ACE%2BENGAGEMENT_MECHANISM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Scale Digital Twin Modeling of Anti-Hypertensive Peptides

# Layer 3 — Which ACE-Engagement Mechanism Can Actually Operate?

### Discriminating candidate mechanisms on a single occupancy axis, using measured potency wherever it exists

**Part 3 of 3** in a multi-scale digital twin of fermented-food antihypertensive peptides.

Layers 1 and 2 delivered a concentration and a falsification. Layer 1 computed the
peptide mixture reaching the gut lumen — dominated by Xaa-Pro dipeptides, with the
lactotripeptides VPP and IPP a minor molar component. Layer 2 established that intact
VPP/IPP in systemic plasma engage roughly **0.018 %** of circulating ACE at the one
measured human Cmax, which is about four orders of magnitude short of what the clinical
blood-pressure effect would require, and that the absorption model fails on tmax in a way
pointing at gastric rather than small-intestinal uptake.

## The premise this layer replaces

The original design for this layer was a **"Systemic ACE Inhibition Potential" score**: a
protein-language-model and docking pipeline predicting binding affinity to ACE, combined
with a predicted IC50 into a personalised efficacy index.

That score cannot be built honestly, for a reason that has nothing to do with how well it
could be computed. It presupposes that systemic ACE inhibition by intact peptide is the
operative mechanism — and Layer 2 showed it is not. A score ranking peptides by their
potency at a target they never meaningfully occupy would rank them precisely, and mean
nothing. Better prediction of an irrelevant quantity is not progress.

So this layer's job changes from **certifying efficacy** to **discriminating mechanisms**.
It asks: of the routes by which these peptides could engage ACE, which ones have
arithmetic that closes? A mechanism survives if measured concentrations and measured
potencies land in the same range. The output is a ranked, falsifiable shortlist — not a
number to put on a label.

## What this notebook claims, and what it refuses to claim

**It claims:** a potency model with its accuracy characterised *per peptide class*, an
audit showing where that model is systematically wrong, and occupancy arithmetic for three
candidate mechanisms reported on several evidential bases rather than the most favourable
one.

**It refuses to claim** that any surviving mechanism is the operative one. In particular,
the step from *inhibiting ACE in the intestinal lumen or at the brush border* to *lowering
blood pressure* is **not established** in anything retrieved for this project. Layer 3 can
show a mechanism is arithmetically possible where another is not. That is a hypothesis
worth an experiment, not a finding.

## Running this notebook

Self-contained: CPU only, no internet, no local files. Every evidence table is embedded
below as a compressed blob and written to `./data` by the setup cell. Runtime is about a
minute; the only fitting is a random forest on 73 rows.

Read in order. As in Layers 1 and 2 the provenance ledger comes **before** any result, so
you know what the numbers rest on before you see them. Section 3 is the one to read
closely: it is where the potency model is shown to be wrong in a direction that matters,
and where the headline number from an earlier draft of this analysis gets retracted.

In [ ]:
# ============================================================
# Embedded evidence. Decoded to ./data -- no download, no internet.
#   ace_peptide_ic50.csv                  - ACE inhibitory peptides, ChEMBL-resolved
#   layer1_peptidome.csv                  - Layer 1's luminal peptide concentrations
#   ace_subsites.csv                      - ACE active-site residues by structure
#   ace_domain_active_site_comparison.csv - C- vs N-domain substitutions
#   ace_structures_summary.csv            - the ACE structures analysed
# ============================================================
import base64, gzip, os

EMBEDDED = {
    "ace_peptide_ic50.csv":
        "H4sIAIfKu2oC/81cbW/bRhL+XqD/gcinFOAR+/6CfFJdWzWcFGqbi5FPAi3RNu8kUkdSSXzoj7/ZJbm7TpNLkGRiw7A4emSK"
        "fGZ2dmdmh+6r/xyrZlPlu6q5GW7zfiibbdlt18PdoYrv3pS7Y/L22NRDnx/OTyRZt936cFHnzbqrDrt6Uw5Vn/s/X+/rZpbK"
        "d/muvaFk3R+6qtzmN/02L/u+vFtvq37T1Yehbpt83+6qzXFXrTe31f5qt663+bbdHPdVM6zvqrLL9+WwuV3v222VX+/Km3XT"
        "NrfHfdms+/bYbSZwU+127q6Guu/h2ldlX23HT8YrOmZw5rC+qptt3dzk48k//nCyylnuOOWKFzRvXuS6YDkd3/kXUpA8P29u"
        "66t6aLu7rNwM9Zt6AOGmrJt+yMrmpm6HqunrJtu0zZuqG+AKWdX8925fZU8XJ6c/5Se/nr74+bmgTCmVU2s56LXqqnYN3+DO"
        "hK+s8rNy18+vLzvQ/ZOTW3daNpTdTTVk45dQQ0z21Ksgg+/Os3829aprh2xFmWH0pyc//vDqciZFFdOFJMoRk4U2Juc5lUYU"
        "xuZUE8cNfgGDF5v/XLdgMbDnLrJ8W/bZvir7Y1dtA+VFQvn8A6T/+iubNAYmjmednBaTJizVWoMijH0G2hDuRebVO7jqelTM"
        "qIBUI1+qjPOoDLhIwWZdgJFnYD56U38rJUxMpbKM53/nl1L7SoKvZ4JcE10opUeGglP4gI9W5prKgktHkTBnbjD69zQ3sV7/"
        "H7L0t9PEIpqaEE/bK4I4U0/AfMQwNQxqy1BN/SqYmlGjwdbMMRSFm1QiFAQUkoQoi0pyGazICVynkBNJ6eblAAUBgySzVktc"
        "lgu38MAaKuNIFQWQAo4zJNOx+v9WoM8jJagwzIyrD9b4PJtNJ5kwpNBy5OUuG6EgYJjOCKNwDRd80GhiVcHVyJE4HwxQEFB8"
        "UBCKTDIYkjJutSi0to4mLyzREEMEE1JLpFAFgKSQKmcFJRJjYfnKyEtqzs2zUdbGSr8i878vRs8ww7LFKipVaAjChDSzUkUO"
        "P2z0d8YsMQpUDjrVXINSGazcT5CW6/b6kydcd+0+O9Q3WVc1cPVN2w3Vuzy7rQ+HY3e3+8dt3Q/1FoRdddzUTZXBXfXHq37o"
        "IDN4MpmAM63IbIMwFxn1bI4CP9McSWD4VdY4T6xhDBOFVaMxIDRmCRYlHz/A7+Mb3kBXKyOezdMDpfYhBvgyhifEWGYLRUeV"
        "Kjf/RyxKKBE3F4Sgzo7PFzFCsQSYSDHylNSHKDMWJRSekE4SjUr0IhI1hBpbTDyFD8VmKAgoax2n2uAG1WEiEFQZZQs+zcpw"
        "ZAkWpcc7EQhYrBX7iO9/Q0ePKhNKGRj4U3QA6xRLsCg9XpUZCGAMusZeB08SUhNjCjprTIArRSxKGM7EqaIEtxqxCEQVVdLK"
        "MbXhBXNTY4CCgMGSamVx58Xl2fskCQs0Wa4Ik9K6Ya9gCZDaGRmI+roL54/QByDfNcaiO8HZH7PaLCxdlBRmWk+Ic4KIRQkn"
        "uaeK46ZP52F8WM6ZhMR+4skdzxkKAk4xinPcKOjVMlbbrIKEpvA+wHzQnGBRQpnSCIUEFNfbz2OmQBhTkLjxiamLgxIwEVEi"
        "Pik4ci15+UiouvKeRq4zvghcIZKWXE9hPCuMdFwjmIgoXDmRErdwvFykmyBMsYLTkeu8DzJhiYgzKzFBUZm+Xj4Opga+HrlG"
        "F5gySakxKviq3w2IWCLiVCOZxF1pls8fB1OppsIKGtNfA1POuLRcFVKPTIV1KXcEExFnm4cpXKNeLB8LVQp2RaX6y3tlFD7V"
        "GllaSPFgIuJkRhI5/Vv++UioWmI1MtVX91N6WtiRaZLSOyxKOHMS4RKV54vgqBDFQ0JLXCw4EnUmTcBExDIpcph0GvYvuSac"
        "6jB6mYseEjARcQo1QnJkrhePhKvb9kKegJcvA1cNOZS1BZ/WGuZ8NQETEYmrNbjT0mVwV2mMEW7dlBNXV31IwEREGcNMwxVw"
        "w6Vl7N5zhTZeiCmFG3v4ApaIODOTwE7gfp+ZaqItLClT7YwV1CWrCZiISMkq4bhWXQaraiYExH7FNDFR56wRixJOmZAJhVtG"
        "/j0SFZRTAkHSlKtS56oJmIg441dp3Gj/z0DVQI4Gsd+0y8Zc6whNwUREokqRd4eX9xqEuJzjwtgh5LEo4dQKNUVeVH+5X803"
        "BZ+JxnK+BxMRqdSN3Hd5urjXWBo69ljsLU3aSxE7TAnRApfp8nEwFYxy5KDw5SOhypmRdrLqJ7pzvrIMsXyv4C0L4UNDeq/i"
        "7dFURrIv1bj2XUW6yhrmI1020tW+FpygqYwyG1vBcKvBZ4tVzufmK2ohVrJj97uCa0I0Ifz4dbmWH8bCqaAwRuUn7f7QHptt"
        "VvdZ5R7dKQegd9122XBbZfW9RsFD2/n+vsOu7Pfl5+1PZ6C7rp2/6Cv6l4EqZGZTjxvIfnowxPUPSv8Uyee1Dib9g1/e/xV1"
        "zeNzBor7NrcRmQ5p7/ZHHo7ggkxlZYHr/+ercNsyeTwiNp0X4YP3b9s1iZ6cZk+Pzb+b9i287+qbugnth5a4m2eEITZphXs3"
        "6aMd1kePI2K+8N4Fxb33s2Ximkm3v7LWt0VOXb/pEoPnlHN2o6Wv7xvygceuvoGLnEV7aZNwhjDQZTYTpD9isg95CTdi3OdH"
        "9pKLxcV859SAkmJYoN2uYQJG6TOHXHboqrrZHK+8Nd/Wwy3YZzK0s63M9sApK+FtXGeugWymiP9o0oWCQUsoQx20oIbLXMyj"
        "lsVJzko/aNk4ZtnD0IcL4/rsxeXz1Uz/206WcPeUc3TjhbvXsBLNd0/d5vuMTIfvbjxjDefY9M9m+tQ/qTLNPtZnpjM0Hx/A"
        "exnD1cDydBU0AIESCxoQ4zbcBM3H768BziEARR4Di1kDSungwryg2tVhAhaEB1CC60NEVsKLMA8YmyrBtyIGLAgP4AtCYs8G"
        "q4swG8DFkhXd+Ig9glF6AD0oju4Rz8OKrmga2UjjZoUIRukBVgfmnxBC1cOroAfLRKoHv3UfwSh9//FAicANEpa/rf4IawSR"
        "1LVcz3pwj5cmYJQeQA9UoPvFeVgpjKXJeKDK7yMGMEoPEPIqjusXpxeXi1UuJ0VMOtAFY14Hnv7HmH+iDuQfNYCsNfu5HW67"
        "9tBn/yo7+NmU2Zuqafdxw1gQnxAzvCf3Vq8vZ45SxLSG+X+hMCHTYSoE+n8YA1a5rhtXtBraT0T3TLinoBAN9T9xhqFUwEcA"
        "AA=="
        ,
    "layer1_peptidome.csv":
        "H4sIAIfKu2oC/41Uy2oVQRTcC/6DH9A25/1YuvAFCiOI6Cpc41UjJgZzBT/f6ojgwgluZobu6Tp1qur09fH6dPHhOC6Ph6uz"
        "Hy/H5cXv1+Hnel2d3fx4/+V4froZX49Xn06fx+W308XHs/Ovh5ub+/eeboPJZ7h7Jyt5ldTwmmqdmtGd1K4x2GxaipfgoU7p"
        "o4aMt4fDw+37twcfLq5/07h/78k2vKcJG04ml1NFDI2ZniXe1W7MOYpmpDuJdgu11h7gm22YTFMqF0+rosKfMt3CiyzXArmO"
        "0imkohEp+KjYA3y8DdWZwulq3tysg2uWtaYyYY2LbVjOLm9BJ9ZWKL0H+A4q5qQObusy8DGPARGIVMuCTI1Dh+QUU4cATalt"
        "sgf4AoA62Vqcw/HWUhs4rc1BEQYYW67kDOzhP2Aqo8ge4sttFI6j41Zp4gbTYdObblUsALMwbEbVUEW7MBmbu6482wYsLeWk"
        "IJIokx4ylVoUCeEQtewFWMRpjSg1hN7Fe7UtOiSQjhiKl0JCnqvVygwN7WJKqEqxImS8oqS7DT+HybDUWKOyDHlASABY1eYq"
        "CKZEBFT12Yxi0I8QWffcz+GCXAlOqAeXBd8wFcZzrCahGIoJdIHGsF0togLNAFH/IN6inr7/PS8bMBxyURjD1cVsYDjIQGjV"
        "CXymgCgqIlqEfFMw9Z2wt6hgIchvIz0dIAdYdCmCVAJoOQQBbK7pK4QS2Srdj/lzYMKRTFXURzJhjTgwcWvg3hAYvGKkDKoY"
        "K4/ExGK4zeVOqq8XLHsKZgzJNHDOAuy6QEBcYQ/Ye/jqKAJh7wqMIzZ1+L+5PlqYZEK5FEBQ4NpiukazHFNAlVoIPJhiCxdR"
        "coJB3jGTGwAQPnJcXxgbHMLR/17jOxT4BcdmvzTEBQAA"
        ,
    "ace_subsites.csv":
        "H4sIAIfKu2oC/81bW2/aSBR+X2n/g7UvTaSBeq621/viONddCmxod7t9iYA4qaXEIDCr7b/fcwbS2A6ODZ2JqIRaKpjznXO+"
        "OVczv52Q29njOM3IIlnOHlZ5OstuIvKQ3o+zW5JmX9NJms8WN9n4McGPpLerhMxvJzfw72z1SFZZOl/M8pv5bEmWq8kyzRPy"
        "mGY3t+kyh3Oym+ksy8fT/Gaczx6Xm3M3b7JkDKfAd5M5HkeePjpd5bO7O/j2NM3HCOjnn+jAVyTubKCyrkt6w2s4bZlms/ki"
        "fSAf/7kmkrmEUibJiL1zjobs3TF8UirCyUAQIuBLw9MTB48KneGHq9Nf4bPC9aVwjk4HV79St0td7r/PxvlqkbiUe+5xC8kX"
        "vU+E+4IEfkB+GVEQTInzJetM0uw2ze6d+8VsNT/+Bb7qIRJmD8nl1YhwyUkg/VeQhCOK1qHv1pjQPH0aDqg9XGvfcPTN68Ce"
        "3KaBeUSQWIUDbg9Y1IvAYOA62eS6gBEKZrIHpffPiEhKwUZUlfgbcMKs8hdZIyl6h/rkOzcI77oKJVO7N0egzi7o/Jr9eZcK"
        "vDvcJpY+YT4lvl80P+8ydLxt83PfIwFQ7HUbcM0EZtcfVDHiKa9EBCGR/Ny2DSBw+W3jA+9KZETshzYNMrw8I0J6QFCwSpEV"
        "HrIiVnYlS8rwUnoNtPCQFjF8zBqYv6IegPExfvMmMB6CYdQuGO67wBVZIqmvtBkssmF0dg3ZQpJAuQ1WCHykxw+4hMWqT/pP"
        "WHhN5uKMcEVbp3qpMyp1w5jSsJjLUFoLmZiiBBQ5SIVihlJ+JUO1PE8XTnCxeEB3LJxaCsCwIuBwrAu/WyIsQtcHQ/GzO3hd"
        "07hwNnebwEs0D99dwuU1oFeAXjYVJwHSjdG9DMQ5BS+45czPiSTIkgHdw6uQSKGeYr5fCpmu0CwJB3I/mEB0HjRdPFrJkDs4"
        "U8CllcwrJ/9K7t+F2HBRBPUb8LJqWbOTRSRYRDRIUMF+JtHBDioOvKCvS4C8XQl2OzAF4yk2T0X2aTr3XxplELNifIZrcf2Z"
        "w2sYYT6oLaClDk+FXI0HPcViT/CACa8Qixl9P0k912WeTI4bpda2nYqjZy1Jbd9yKlHpfs0rv2NfB6yF2MJkqbEziWrd1ClI"
        "05hWOEjmxMlmWQdI95hs8gkkQlYqHU27p1Vfw7rQcVQCgEEcxXnA9vzn6+EI+OLzR5vOWHfYz3ccBAfaBcye4i9b2pLmAd1o"
        "rizavrGtC7fxE0Il03ckjEVo6+6Wi9ntKHi17DIov9JTMHRMFQBbdzZgByps3dSarl/qisqzyY3d2l2uKcEhaHFrnMCmE4vx"
        "IFB1nOBYzMbSnl0EdTFuyjoAQttBhLGyZ4OtLb/QmkPeskaLaDQE9YF5LvYrBdG6s+XMnsKFScPWi6h0V0/3TVP005fzcmf9"
        "IR6Q6Xie7zK31i2sLmeK0wU4+gkHlCJK+m4VhysC4btZCxzlYq6QrZSqTu5Nii1lq4ZyLqiOZE0CKZfQBf29F5NHk2JLQbjY"
        "hXo2rd62hoamEkuFkVXOtaihnyc5ehIM1ZMMrTGh/VBaYsttyzjlYFyghtLB2LOqfVN+1oFRD/psUbTteop3PQE4RhZvSss1"
        "jR587otDXcq/n3HQLvfI+UefwNv5OAcg47w+QnuVvIBn7XDiTvdPL02l3j2IPSTWR1m/EtxbH7m10cMTA1zuslKr3frQutYN"
        "z4QCrM/2BLp911gdku6me6tQBd0mlnR7yKjLUPrEvTy2w51iojKF3NE0beIYC/aV0S6NVtKXXNNyL1/U5gSBtVHs73Fk+8XO"
        "Ztem9ofedqmnG1Aq95BStyCDpg5i8wuyijhSz2NaqgcjZ+dk/nW2nH9Ns3TqwLnzZJ6nt4lzfuUcQeMgXK+Tf5snxzvscKQO"
        "0dHFs3yU/KPyt0/un6SZl9W88qlMOwPikTgK45MwPoVAHJ2axVS7k9ONgllZ27dFWhZ6NjYrDbthHkChy1jd8k5nomgQDi7N"
        "it6+woLMZ4FUW3eUKEsXA8aNqtfG0EQrXjcbD3AyO4hObAhe76uLQQlTIIlPzi0wtXE9VjN8ZXrCFF3DnT0JjQesdrvBemhc"
        "QzMeR3DZzKF6xEqqVOGs79inMI7+AoNENvzUtNgNt1z/TRkh19jOdGQNrWQXjuEt8GsHoroDjP40K3hdkUC4qRScgmt3/A7u"
        "sBHcId4JXZpsH70KokD4JQj/A1690LjaukCCVkBCiCrNXnVp98V4oNe1JHBP0qBu9Cn1pCH6AMqOzF+55+c7tgtX2t19C+T6"
        "vtp/Ibgm9uiKCowQGs8NuheCukJUnkLBIh/JbjgTXV8Ax8DnLKgj+vrpPiijzZMNYpZQZXJ7mtxYH1ILtSHkXIE5tyBQP1/x"
        "8irRgR+Vx0D9zxFIn44Xk9l/38YP4yzNEufobjG+f0yyPHSycZ7+mzjLfLGa4tNrx+X5TMMoG2up4toZ5VcekTMDqjRwaXrQ"
        "jL0RqLbP6m+JC5uH9qHuPWVvhLSwntgKKNDjmzeBU5zFNDwW93YUa/8kzLYwr+ch0CS+lQGLE5u6fR9EiFP6BmhK45yG4bLS"
        "vUIDKBENfy+CUuRkOJw406/w1hmu3xz9dvGw6lw8fOv0klVnuJjpV7S413/3vi07Vw/J0/9XotrmR0pLJ+7kyeIxzcYPTuHT"
        "zuzOWcvY/LrJeYrPmx9POfdggUzPKMC+AzK8HkBs+h6KAf1GJcZdn3nSfx6Yw5/3d8lk2aXM5f6xFV1LwdKcrpDkDk/X0q01"
        "pyvkV8zn/fDgFC6FclMK60d7CVQvoO9V7yw4HHWLqcKctvgTg4PzbHFZY05Vph0bxhdaXfdw1C1lDXP6cn1zD823pbWBOWUF"
        "07f24O5sYTFmTlnoucihhadiC2BQU09nn/iCHZi6xS2bOXV90PbgVC093GpO10D88I39H4ULJmXwPwAA"
        ,
    "ace_domain_active_site_comparison.csv":
        "H4sIAIfKu2oC/+2a32/bNhDH3wfsfyDy0gQ7B/wtcnlS3axpkbqCna7LXgQlVhMBtuRZdoD+9zvSkq206ZIm8p4IBOZZkh3p"
        "8yXvjnceptNqnhVlui6LxbJapYuqhmF7cDG9Spd5Xa7nKfto9O4EHiym6xxGD3189MDH+VCPdifajxfTvFwV19kMhnGKf9Oi"
        "XqUx1Pk/67y8ztNsVtyUc7wmzW6WeV5Dvb6qi1W+uZv2Df845IDfOivqoqwWy2K2OY+Hxn8lsTB0ewn++0W2wiuyVarP1Gd3"
        "7HWSXKUyTt67N4vbql7cFmVxncphrNMR3jHg7RR32czf0jxf3VZTuC5W2aqoyl9/0ToCzeHicgxaglAwOR3DH9mszoEdcwYX"
        "S3xQaI50X/2JjXkwqxACqdeLfIkEC/fNpPpCHLXD0QDv4YhU5aoi7rkOh+0Bwi2pb7NlPiW7Zye3eXb3lWSral6Tw/GHyRtC"
        "j4Ug8RGQMseL69VgGJNFViyL8uaEFOU0X+SlU2L2lVxX5ZdiOcdvvPpKRnk+neXzrBx8Xpf19S3Z6uFuLmHccEYEHWjJyF1N"
        "tOQDxgU/gOTNa3+r5Df/CG5AAXBwzHFwtN2AhE/Ip7JIcOo033dCkg/v3vzOuJLUKNm+jaSwXEaIO2KgNcSTEWgDwt7DLcSP"
        "cXdfA++n87YUjPK8TQTKQHweNxTpsQ28++YdcQ0M3UY8SXDUYCO4ONvNcGmCQ+kbuEHQAt6ef8LRgLVoXm6BKxmA9w3cImgJ"
        "8fgtjmhTes+JqwC8Z+CSAsPExKUozNmMedvTZMdUBSfeO3GJxC2cn6JPcTZT3m6Js0C8d+LoxWXjxZ3NupkhPVYmIO8bOW58"
        "GO58PHJtvWtxWUuLXDd+ZcJekcOEvTqCrv8O6F+A3mD45M0M5y5hdFni5S6EMh5me+/IFaKmcDFOcMSZb423PWd6bALx/okb"
        "JC3gw+kFcEGBu7A5TraTnMqAvG/kRgM3DF36CEfEr6y3m0lum7xlwp1H5+jRd5Y/8dBLwP9k/BazFKEEnL2bgNAUhGDebvDr"
        "xsccTJgLqED+LgdX+Ij4qORmWa0XRyfbWHuwC7snQaR+RbIokvR1MKEZisS93YgU6cdE6kgTBOlDELdS1KbuLnCNCCG83Qgi"
        "1eOCHEwEnhOIoyoHiGye48GgTR/a4AJRulksArWR3cWybY88KMB3NZ8gwbMkUD6YuP2BwC2bEGZX/cGgIkK5rWfgBoEbCn/G"
        "5zgicFd7ONuVIiLxX/vib80A/ifAY/pkmvTJUh+dO+mTjB5Pn9pM6eCJV30TGYJuz9MNMyojfSFJ4B5DNEWlVjf+lAD+43NB"
        "oz40stw7s83akqiR6q4tYV+i0UlIv/almvYrKjk7xRG9o7adOiFu6OVP5V9h6TxHBEZpBJJy3/CU1PrkwNlNNyhUDvdA3CDp"
        "TTdIMswEDPPEt/Xx0NTvHTlTHrULCpJh1ovJWDdA8OBp/g8RNMJnzbx32bDtZlLc9JpJhZD8LI04RW2Ub5VKjlmVFd5uG0mt"
        "a9pVbcPq6Ik87iukinw2JLUEKZS32+5GaG7sFT9D56TQOZ1fTnDEAI3OydltISrw3zf/CLlzP+UVsyAt7U5/yZ4SHLhTJGzU"
        "9iCOQVE2tSvlIoS9V7sS39UMQ3tvX0pwhkpo30JSXKAS0v/6vd0zcx72Dr0jFz4iuIq54riPsPpexVy82DOF/tFL1JE+XiTj"
        "j6gObjBs5O3GNSkd6hf9I1c+CrhaneIRIr/Xs9t2kEKqtC/+mCrxTcVOcQuKdip2yP/RX54F4j9N3Pho62e8oEicPTTj99i7"
        "CyvlGbr9C6Yz1+4/PAAA"
        ,
    "ace_structures_summary.csv":
        "H4sIAIfKu2oC/72Yb4/iOBKH359038GaV0Fy2NjO30NIl4Z0w4omOaD3mh6dUBoyTSRIUBLmtne+/JWdBHACszvavX5BdxyX"
        "Hbvy1K/KOWxeV/EGZ1Ge7o5FnCYrF++jYpvCvdWXLIqgaxeFebTahEWEi7jYRXiT7sM4WYV5Hr8l0Qbs0v3qmMSHLC1WWZi8"
        "RbhuBYTalFQ3+Z0IBq63MDyHZpys40O4W8XJNn6NizTDp6tVEu4jnKx+S1Z5XEQ5hot1mmabOAmLOHlb1ZPB0uPN8YrBLn4L"
        "k80qLNJ9jsPdbrWNiihLebvqyzEfkhThujhNs9JdfO1enh6zdfT3vxHfNjHtavhZzcJ3rHUpxVTTmKpRVbPwIHvPi3CH5kV2"
        "XBfHLELpFzQ67sMEuclbDEtO8jhBgzT5GmV8mchLfnvfRwhurtP9YRf9iv4bF1u0i8EuBRftuniglh7Hpk5VQpmGXTwJZvhs"
        "gwl+mDzphCilw/9BNGJ2/gnrdHtoNJ4zm9U9jm2LDr3usU49DhU9lpi96zNoEG4mWhRaJnHhOcs+/AbjqddDL9P+y3g6QGN/"
        "Ksz6n6dU/azMOypRB+7szn9eqkwNRt50Cb1+sJz8R52ok+W8hwaT/mA08WfjoceHY2LhxXJmUE0hhBqdHoINMVtXHNvplCs1"
        "mOIYNjSEHeN2vOVOXGaAncHtYGoDvEAIbF+MMgg3JHY5IfeQcA1vTalNFNuuDLkjuAeEHTGpYplW5+Q87rUego3ohgUzmLRs"
        "GYTy2bnhL+7EIDZfFCtbzNZgGN/J3Jsxw1AcU+vgT7PB/A5FSZG9I84SUvb7wfi+g5HoGIZFiNxgjB6y8LD91wRBMIYQemEP"
        "Jcf9a5RxZPbh4RBtUJGipyQOIA5Q+QbR1zhE2Tp/XR3SHUCVreA5cfG+CoH35JOA1/1/wKtMIeS+Rp3rqE6f4aeuw+w1/fU9"
        "3IVJnMCQL1n4tof1wcbEYJTXT+3coJl0HesmzvQmzqaL3cGi65fXwAtvPC84zrAu96FP1XANXt7Hm1Sl6iaCVaqvcEMdqm+7"
        "4zo9vIN8pXkkhvbdgbdwF16JvMQ/7LM/rbEHKqciRJqc2w3MW8SeOW/iewGsHAJSqFxC2sTN/Vjcnl7uG7hZnDcdYFOJiQeV"
        "4AFlxTZCIYxWt+8HYIuDBkhssuMbWoeHQogcGAi7rSASckIRr4+7MIP7ZzjH6vqMZyTwvArl48DHp5lvy6d+CzhyizdCxOTd"
        "OVwz+qOESUDBNH2iKkx99GYDN1j4MPjRW4yWE5WLKdgsJx01WM5m/uSKoFLcVs0TdjJbMoQSaRJbMnaSasraKMmmpNCXKizj"
        "CbR8KJ7UH4D8dalx5tMUemipmqGCNEo6KHjLQQK8MkHPngMXJL4FFwW4Zs8M1/03tYzc1LKbUkbE1DwzV1ooWjp06aBln6AF"
        "AqR8U8gc01lHrVLxZ4XMeEL+9lm586YvywnIU6fFi8TeJ0wcLEPxneR8JR+bkO1IK+dKPEpUyDhKYF1kz2aWbefwi0zNHAIz"
        "VD060fgijBao7jzQiQEtqjVSugQnZ+VD4TRHxr8x6TLrBCexDAZ0Elsl5DJb55eUbpvZWiWoJYioZrZVdfqw3rAARQyL7lWy"
        "7xc2Ts9GfyBTX8nHNyWVT9+d80qTcSveEnQbDtAdmA/9kffsCgH0ph7iVSgXvjuf9e+A4wEE53jYQ+PHYX/8OB66L/7EE7P0"
        "L/b1ffB7yBuC6GKqwmPcqTcc+xMIBqORtGXiv5PC/yjkTcltBM2J65uZvhEnrdq0rkYlqjlkHyu5A3OK2UVFwDSbK66pQhEK"
        "Xp63Wc4BN0j0ZZJPKqjHV6CeogrqxvkJas+KY6appk6w27trHp8oFo51aiJ1eENAseVUrJonwJmjCYqNusc49+iC4svjk8Gk"
        "45OoROUK8sGfiOOUJ3L4nzpCNcvRHyg8MNExR5VRhQndhvl1cIdRH5wYHIhAUCsxBWU1hF7yONCgwURjNNMdE3qq+oIxArNp"
        "Fd2Go1C75h78yR0pJtAdG8ZYNdyOop/CA5zLvVqyzRjMVsUKBx3ioKHRANdH03zPaSbnAsJiJc5cou2/BOer+OLwkCIlSdHp"
        "e0nnFsPOTYbJLYSZi0f+SJyYTN3FLcxkZn/nUPQjFGq4+UI/tiLUB66JeWl2cWKhjsmzLsgTJEGC28fhK6/uVnoN0WGb5odt"
        "nAABkI4O0aGINxG6H8tvmTniLTPvHt8agBQoMHUoVQs4MPGXf12m3FNUXVIBHezWywdl4w/u+u6QM8LP1WXzgesZ4BA86P2F"
        "t5i1E3HgPfSHY2W0HM5AtUR3B/56s9+TqpupHZ4MFe03hYIcslY1+8r9vQOaOvzjAoZCXlIqSVskpZLUTVIqKAeZwxSd0iuy"
        "ddY9oZU6jGF1oxZOSbYkQQN5ZIYNsmVdEUFmOtCw6zRugdkp99swgVXldM2ALehV7nc02ILTVN6GVj7psDm93NzswdBgDK2L"
        "B2ZAj145BLagwxakCOQB8bER6AY/w5nMPAegYULUQQBC9GkqsaoAvDjytw/8rbi7C4LXa18BegHmXagOK0V8mUZBp3G7Wiw6"
        "dWt46gOmXIFfErRPN9FuB/uN+QcKOCQK53Vw8lPYFs+5r/fnT5P7k1wGng4K+Q2SPlWViwsIBp77y3+d8n9LbTG126UpoAxl"
        "YHVM0zWoODWtQssgFjekrapVOrUJQ5MbkjIm4EKxqHntC8CtL1RSOSqVwYtZAFFlmlWEwKuAudufV+UqWC5o+SqIYhpmOR0P"
        "GOf83aEuqqVKfDzxIJJMh7VOkeAIWD+sgZWh5FAwq54CYmKK78AQO4TqYOQ0IgSA/dAI+R8U+bXdshkAAA=="
        ,
}

os.makedirs("data", exist_ok=True)
for _name, _b64 in EMBEDDED.items():
    with open(os.path.join("data", _name), "wb") as _fh:
        _fh.write(gzip.decompress(base64.b64decode(_b64)))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from scipy.stats import spearmanr

try:
    display
except NameError:
    display = print

os.makedirs("figures", exist_ok=True)
np.random.seed(0)

ic50 = pd.read_csv("data/ace_peptide_ic50.csv")
peptidome = pd.read_csv("data/layer1_peptidome.csv")
subsites = pd.read_csv("data/ace_subsites.csv")
domains = pd.read_csv("data/ace_domain_active_site_comparison.csv")
structures = pd.read_csv("data/ace_structures_summary.csv")
print(f"evidence loaded: {len(ic50)} potency records, {len(peptidome)} luminal peptides, "
      f"{len(subsites)} active-site residue rows, {len(structures)} ACE structures")

evidence loaded: 74 potency records, 17 luminal peptides, 122 active-site residue rows, 9 ACE structures


---

# 0. Provenance ledger — what the potency model is built from

The training set was assembled by querying ChEMBL for bioactivity against human ACE
(target CHEMBL1808) and resolving records to peptide sequences. That resolution is the
first thing to be honest about: of 1,111 retrieved IC50 and Ki records, only **88 (7.9 %)**
could be mapped to a definite peptide sequence, collapsing to 73 unique sequences. The
1,023 unassigned records are dominated by synthetic small molecules — 416 contain sulfur
and 96 phosphorus, i.e. captopril-like and phosphinic inhibitors, not peptides.

Two consequences follow, and they shape every number downstream.

**The set is small and skewed by length.** It is mostly dipeptides. A model trained on it
knows dipeptides well and longer peptides by extrapolation, which is the opposite of what
a general peptide-potency predictor should look like.

**It is almost entirely IC50, not Ki.** IC50 depends on assay substrate and concentration,
so pooling across studies introduces a spread that is not biological. The composition
below is printed rather than described so the skew is visible before any model is fitted.

In [ ]:
# ============================================================
# 0. Provenance
# ============================================================
AA = set("ACDEFGHIKLMNPQRSTVWY")
train = ic50.dropna(subset=["sequence", "pIC50_or_pKi"]).copy()
train = train[train.sequence.apply(lambda s: isinstance(s, str) and set(s) <= AA and len(s) > 1)]
train = train.drop_duplicates("sequence").reset_index(drop=True)

comp = (train.assign(length=train.sequence.str.len())
        .groupby("length")
        .agg(n=("sequence", "size"),
             pIC50_min=("pIC50_or_pKi", "min"),
             pIC50_median=("pIC50_or_pKi", "median"),
             pIC50_max=("pIC50_or_pKi", "max"))
        .reset_index())
display(comp.round(2))

n_ki = int((train.standard_type == "Ki").sum()) if "standard_type" in train else 0
print(f"unique peptides                : {len(train)}")
print(f"  of which Ki rather than IC50 : {n_ki}")
print(f"pIC50 range                    : {train.pIC50_or_pKi.min():.2f} - "
      f"{train.pIC50_or_pKi.max():.2f}  (median {train.pIC50_or_pKi.median():.2f})")
print(f"dipeptide fraction of training set: "
      f"{(train.sequence.str.len() == 2).mean() * 100:.0f} %")

for col, label in (("flag_nonhuman_source", "non-human enzyme source"),
                   ("flag_cell_or_tissue_based", "cell- or tissue-based assay"),
                   ("flag_assay_type_not_binding", "assay type not a binding measurement")):
    if col in train.columns:
        print(f"records flagged, {label:38}: {int(train[col].fillna(False).astype(bool).sum())}")
print("\nFlagged records are RETAINED -- removing them would shrink an already small set.")
print("They are a source of the spread quantified in Section 2, not a hidden exclusion.")

,length,n,pIC50_min,pIC50_median,pIC50_max
0,2,52,1.77,2.78,7.20
1,3,7,2.73,5.10,5.70
2,4,12,2.16,3.14,5.70
3,5,2,5.27,6.24,7.22


unique peptides                : 73
  of which Ki rather than IC50 : 1
pIC50 range                    : 1.77 - 7.22  (median 3.03)
dipeptide fraction of training set: 71 %
records flagged, non-human enzyme source               : 3
records flagged, cell- or tissue-based assay           : 2
records flagged, assay type not a binding measurement  : 6

Flagged records are RETAINED -- removing them would shrink an already small set.
They are a source of the spread quantified in Section 2, not a hidden exclusion.


---

# 1. The potency model

Peptide ACE potency is predicted from physicochemical descriptors of the sequence:
hydropathy and residue volume overall and at the C-terminal positions P1, P2, P3, counts
of proline, aromatic and charged residues, and a flag for whether the C-terminal residue
is one of the residues ACE's S1' and S2' subsites favour (Pro, Trp, Tyr, Phe, Ile, Leu).

The positional terms carry the mechanism. ACE is a carboxypeptidase: it engages the
substrate's C-terminus, so the last two or three residues dominate binding, and a
descriptor set that averages over the whole sequence throws away exactly the signal that
matters. This is why a simple positional encoding is competitive here.

**On protein language models.** An earlier version of this analysis ran a head-to-head
against ESM-2 embeddings (`esm2_t12_35M_UR50D`, 480 dimensions) on a 42-descriptor
version of this feature set. The descriptors won: R²cv 0.723 against 0.562 for ESM-2, with
concatenation no better than descriptors alone at 0.557–0.608. That result is reported here
rather than quietly dropped, and it is not surprising on reflection — a 73-row training set
cannot constrain a 480-dimensional representation, and a general-purpose protein embedding
has no reason to encode the specific positional grammar of a carboxypeptidase active site.
The ESM-2 comparison is not rerun in this notebook because it needs downloaded weights and
this notebook runs with internet off.

In [ ]:
# ============================================================
# 1. Descriptors and the potency model
# ============================================================
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5,
      "G": -0.4, "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8,
      "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
VOL = {"A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5, "Q": 143.8,
       "E": 138.4, "G": 60.1, "H": 153.2, "I": 166.7, "L": 166.7, "K": 168.6,
       "M": 162.9, "F": 189.9, "P": 112.7, "S": 89.0, "T": 116.1, "W": 227.8,
       "Y": 193.6, "V": 140.0}
CTERM_FAVOURED = set("PWYFIL")     # ACE S1'/S2' preference


def descriptors(seq):
    """Physicochemical descriptors. Positional terms are indexed from the C-terminus,
    because ACE engages the substrate C-terminus."""
    n = len(seq)
    return {
        "length":         n,
        "hyd_mean":       sum(KD[c] for c in seq) / n,
        "hyd_sum":        sum(KD[c] for c in seq),
        "vol_sum":        sum(VOL[c] for c in seq),
        "vol_mean":       sum(VOL[c] for c in seq) / n,
        "n_pro":          seq.count("P"),
        "n_aromatic":     sum(c in "FWY" for c in seq),
        "n_charged":      sum(c in "DEKR" for c in seq),
        "cterm_favoured": float(seq[-1] in CTERM_FAVOURED),
        "has_xpp_motif":  float(n >= 3 and seq[-2:] == "PP"),
        "p1_hyd":         KD[seq[-1]],
        "p1_vol":         VOL[seq[-1]],
        "p2_hyd":         KD[seq[-2]] if n >= 2 else 0.0,
        "p2_vol":         VOL[seq[-2]] if n >= 2 else 0.0,
        "p3_hyd":         KD[seq[-3]] if n >= 3 else 0.0,
        "p3_vol":         VOL[seq[-3]] if n >= 3 else 0.0,
        "nterm_hyd":      KD[seq[0]],
        "nterm_vol":      VOL[seq[0]],
    }


def design_matrix(seqs):
    return pd.DataFrame([descriptors(s) for s in seqs], index=list(seqs))


FEATURES = list(descriptors("VPP").keys())
X_train = design_matrix(train.sequence)[FEATURES]
y_train = train.pIC50_or_pKi.to_numpy()


def new_forest():
    return RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=0)


cv = KFold(n_splits=10, shuffle=True, random_state=0)
y_cv = cross_val_predict(new_forest(), X_train, y_train, cv=cv)
ss_res = float(((y_train - y_cv) ** 2).sum())
ss_tot = float(((y_train - y_train.mean()) ** 2).sum())
r2_cv = 1.0 - ss_res / ss_tot
rho_cv = float(spearmanr(y_train, y_cv).statistic)
rmse_cv = float(np.sqrt(ss_res / len(y_train)))

model = new_forest().fit(X_train, y_train)
print(f"{len(FEATURES)} descriptors, {len(train)} peptides, 10-fold CV")
print(f"  R2   (cross-validated) : {r2_cv:.3f}")
print(f"  Spearman rho           : {rho_cv:.3f}")
print(f"  RMSE  (log10 units)    : {rmse_cv:.3f}   -> a typical prediction is off by "
      f"{10 ** rmse_cv:.1f}x in IC50")

imp = (pd.DataFrame({"descriptor": FEATURES, "importance": model.feature_importances_})
       .sort_values("importance", ascending=False).head(8).reset_index(drop=True))
display(imp.round(3))
print("The C-terminal descriptors dominate, which is what a carboxypeptidase should give.")

18 descriptors, 73 peptides, 10-fold CV
  R2   (cross-validated) : 0.738
  Spearman rho           : 0.866
  RMSE  (log10 units)    : 0.687   -> a typical prediction is off by 4.9x in IC50


,descriptor,importance
0,cterm_favoured,0.504
1,p1_hyd,0.205
2,vol_sum,0.088
3,nterm_vol,0.043
4,vol_mean,0.028
5,hyd_sum,0.022
6,hyd_mean,0.021
7,n_aromatic,0.020


The C-terminal descriptors dominate, which is what a carboxypeptidase should give.


---

# 2. What the model is for, and the test that matches it

A cross-validated R² is the wrong number to select on here. Cross-validation shuffles
dipeptides and longer peptides together, so most folds predict a dipeptide from other
dipeptides — easy, and not the job. In deployment the model is asked to score peptides
that Layer 1 produced, whose length distribution it does not control.

So the model is tested the way it will be used: train on the dipeptides alone, predict
every longer peptide, and see what happens. This is a deliberately harsh test — nothing
about tripeptide binding is in the training data — and it separates two failure modes that
CV conflates. A model can **rank** correctly while being **systematically shifted**, and
for occupancy arithmetic the shift is what matters: rank errors reorder a list, but a
one-log shift in absolute IC50 changes an occupancy from negligible to substantial.

In [ ]:
# ============================================================
# 2. Length extrapolation: dipeptides -> everything longer
# ============================================================
is_di = train.sequence.str.len() == 2
fit_di = new_forest().fit(X_train[is_di.values], y_train[is_di.values])
y_long_pred = fit_di.predict(X_train[~is_di.values])
y_long_obs = y_train[~is_di.values]

res_l = float(((y_long_obs - y_long_pred) ** 2).sum())
tot_l = float(((y_long_obs - y_long_obs.mean()) ** 2).sum())
bias_long = float(np.mean(y_long_pred - y_long_obs))
print(f"train on {int(is_di.sum())} dipeptides -> predict {len(y_long_obs)} peptides of length 3-5")
print(f"  R2               : {1.0 - res_l / tot_l:.3f}")
print(f"  Spearman rho     : {float(spearmanr(y_long_obs, y_long_pred).statistic):.3f}")
print(f"  mean bias (pIC50): {bias_long:+.3f}  -> predicted IC50 is "
      f"{10 ** (-bias_long):.1f}x {'WEAKER' if bias_long < 0 else 'more potent'} than measured")
print("\nRanking survives extrapolation; the absolute scale does not. For a ranked shortlist")
print("that is acceptable. For occupancy arithmetic it is not, which is why Section 3")
print("measures the shift separately for each peptide class rather than trusting one number.")

train on 52 dipeptides -> predict 21 peptides of length 3-5
  R2               : 0.392
  Spearman rho     : 0.813
  mean bias (pIC50): -0.586  -> predicted IC50 is 3.9x WEAKER than measured

Ranking survives extrapolation; the absolute scale does not. For a ranked shortlist
that is acceptable. For occupancy arithmetic it is not, which is why Section 3
measures the shift separately for each peptide class rather than trusting one number.


---

# 3. The audit that changes the answer

Section 4 will compute how much ACE each candidate mechanism engages, and to do that it
needs an IC50 for every peptide Layer 1 produced. The obvious move is to predict them all
with the model from Section 1. **That is a mistake, and this section is why.**

Seven of Layer 1's seventeen luminal peptides already have a *measured* IC50 in the
training set. That gives a direct, out-of-sample check on exactly the peptides the
occupancy arithmetic depends on: refit the model with one of them held out, predict it,
and compare against its measured value. If the model is trustworthy on this class, the
predictions should scatter around the measurements without a consistent direction.

They do not. And the direction of the error is the one that would manufacture a positive
result.

### Why this matters more than a fit statistic

An earlier draft of this analysis quoted a combined luminal ACE occupancy of **0.877**,
attributing 90 % of it to Xaa-Pro dipeptides. That figure used model predictions for every
peptide. Its two largest contributors, Phe-Pro and Glu-Pro, are dipeptides with no measured
value — precisely the class this audit shows the model over-scores. The number below
replaces it, and the correction is not small.

In [ ]:
# ============================================================
# 3. Out-of-sample audit on the peptides that actually matter
# ============================================================
measured = train.set_index("sequence").pIC50_or_pKi
overlap = [s for s in peptidome.peptide if s in measured.index]
print(f"{len(overlap)} of {len(peptidome)} luminal peptides have a measured IC50: "
      f"{', '.join(overlap)}\n")

rows = []
for seq in overlap:
    keep = (train.sequence != seq).to_numpy()
    held = new_forest().fit(X_train[keep], y_train[keep])
    p_loo = float(held.predict(design_matrix([seq])[FEATURES])[0])
    rows.append({"peptide": seq, "length": len(seq),
                 "measured_IC50_uM": 10 ** (6 - measured[seq]),
                 "loo_pred_IC50_uM": 10 ** (6 - p_loo),
                 "log10_error": p_loo - measured[seq]})
audit = pd.DataFrame(rows)
audit["fold_too_potent"] = 10 ** audit.log10_error      # >1 = predicted more potent than measured
display(audit.round(2))

bias_by_len = audit.groupby("length").log10_error.agg(["mean", "size"]).rename(
    columns={"mean": "mean_log10_error", "size": "n"})
bias_by_len["fold_too_potent"] = 10 ** bias_by_len.mean_log10_error
display(bias_by_len.round(3))

DI_BIAS_LOG10 = float(audit.loc[audit.length == 2, "log10_error"].mean())
print(f"dipeptides : model is {10 ** DI_BIAS_LOG10:.1f}x too POTENT out-of-sample "
      f"({int((audit.length == 2).sum())} peptides)")
print(f"tripeptides: model is {10 ** float(audit.loc[audit.length == 3, 'log10_error'].mean()):.1f}x "
      f"off, direction mixed ({int((audit.length == 3).sum())} peptides)")
print("\nThe model is accurate on the tripeptides it was effectively built around and")
print("systematically over-optimistic on dipeptides.")

# Name the offenders from the data rather than by hand, so the claim cannot drift.
rank = peptidome.sort_values("mean_uM", ascending=False).reset_index(drop=True)
rank["abundance_rank"] = rank.index + 1
worst = audit.loc[audit.length == 2].nlargest(2, "log10_error").merge(
    rank[["peptide", "mean_uM", "abundance_rank"]], on="peptide")
for _, r in worst.iterrows():
    print(f"  {r.peptide}: rank {int(r.abundance_rank)} of {len(rank)} by luminal concentration "
          f"({r.mean_uM:.1f} uM), measured IC50 {r.measured_IC50_uM:.0f} uM, "
          f"model {r.fold_too_potent:.0f}x too potent")
top = rank.iloc[0]
print(f"  most abundant luminal peptide overall: {top.peptide} at {top.mean_uM:.1f} uM")
print("\nThe worst-predicted dipeptides are among the most abundant, and they are measured")
print("at hundreds of micromolar -- an order of magnitude weaker than the model predicts.")
print("Abundance does not buy them potency, and a model-only calculation would have")
print("concluded that it does.")
print("\nSection 4 therefore uses measured IC50 wherever one exists, and applies the")
print("dipeptide correction above to the predictions where none does.")

7 of 17 luminal peptides have a measured IC50: GP, VP, IP, VPP, FPP, IPP, AP



,peptide,length,measured_IC50_uM,loo_pred_IC50_uM,log10_error,fold_too_potent
0,GP,2,446.68,26.35,1.23,16.95
1,VP,2,416.87,19.67,1.33,21.20
2,IP,2,128.82,42.84,0.48,3.01
3,VPP,3,8.00,6.94,0.06,1.15
4,FPP,3,77.98,5.78,1.13,13.48
5,IPP,3,5.00,11.47,-0.36,0.44
6,AP,2,50.00,30.29,0.22,1.65


,mean_log10_error,n,fold_too_potent
length,,,
2,0.813,4,6.499
3,0.277,3,1.892


dipeptides : model is 6.5x too POTENT out-of-sample (4 peptides)
tripeptides: model is 1.9x off, direction mixed (3 peptides)

The model is accurate on the tripeptides it was effectively built around and
systematically over-optimistic on dipeptides.
  VP: rank 3 of 17 by luminal concentration (42.4 uM), measured IC50 417 uM, model 21x too potent
  GP: rank 1 of 17 by luminal concentration (105.7 uM), measured IC50 447 uM, model 17x too potent
  most abundant luminal peptide overall: GP at 105.7 uM

The worst-predicted dipeptides are among the most abundant, and they are measured
at hundreds of micromolar -- an order of magnitude weaker than the model predicts.
Abundance does not buy them potency, and a model-only calculation would have
concluded that it does.

Section 4 therefore uses measured IC50 wherever one exists, and applies the
dipeptide correction above to the predictions where none does.


---

# 4. Mechanism discrimination on a common occupancy axis

Three mechanisms could in principle connect a fermented-milk peptide to ACE inhibition.
They are usually argued about qualitatively; here they are put on one axis so they can be
compared, and the axis is **fractional ACE occupancy**, because that is what determines
whether an enzyme is inhibited.

| Candidate | Where | Concentration from | Potency from |
|---|---|---|---|
| **M1** systemic tripeptide | circulating plasma | measured human Cmax, 897 pmol/L | measured VPP/IPP IC50 |
| **M2** luminal tripeptide | intestinal lumen / brush border | Layer 1 | measured VPP/IPP IC50 |
| **M3** luminal dipeptide | intestinal lumen / brush border | Layer 1 | model, bias-corrected |

### The arithmetic, and one thing it is easy to get wrong

For a single competitive inhibitor, fractional occupancy is `C / (C + IC50)`. For a
*mixture* — which is what a gut lumen is — occupancies **do not add**. Each inhibitor
competes with all the others for the same site, and the correct combination is

    S = sum over i of  C_i / IC50_i
    fractional occupancy = S / (1 + S)

Summing individual occupancies instead can exceed 1, which is a useful sign that the wrong
formula is in use. With seventeen peptides present simultaneously this is not a technicality:
the naive sum for this mixture comes to more than 3.

Each mechanism is evaluated on several **evidential bases** — measured only, measured plus
corrected prediction, and prediction alone — so the reader can see how much of the answer
is measurement and how much is model.

In [ ]:
# ============================================================
# 4. Occupancy for each candidate mechanism
# ============================================================
IC50_MEASURED_uM = {"IPP": 5.0, "VPP": 9.0}     # PMID 7790570
CMAX_HUMAN_uM = 0.897e-3                        # PMID 17374660, plasma IPP
IPP_BIOAVAILABILITY = (0.077e-2, 0.38e-2)       # measured range, for context only

pep = peptidome.copy()
pep["pred_pIC50"] = model.predict(design_matrix(pep.peptide)[FEATURES])
pep["pred_IC50_uM"] = 10 ** (6 - pep.pred_pIC50)
pep["measured_IC50_uM"] = [10 ** (6 - measured[s]) if s in measured.index else np.nan
                           for s in pep.peptide]
pep["ic50_source"] = np.where(pep.measured_IC50_uM.notna(), "measured", "predicted")

# bias-corrected IC50: measured if available; else prediction shifted by the
# class-specific out-of-sample bias measured in Section 3
pep["corrected_IC50_uM"] = np.where(
    pep.measured_IC50_uM.notna(), pep.measured_IC50_uM,
    np.where(pep.peptide.str.len() == 2,
             pep.pred_IC50_uM * 10 ** DI_BIAS_LOG10, pep.pred_IC50_uM))


assert (pep.loc[pep.ic50_source == "predicted", "corrected_IC50_uM"]
        >= pep.loc[pep.ic50_source == "predicted", "pred_IC50_uM"]).all(), \
    "bias correction must make predicted IC50 WEAKER, not stronger"


def mixture_occupancy(conc, ic50):
    """Fractional occupancy of one site by a competing mixture. S/(1+S), not sum(C/(C+K))."""
    s = float(np.sum(np.asarray(conc, float) / np.asarray(ic50, float)))
    return s, s / (1.0 + s)


di = pep.peptide.str.len() == 2
tri = pep.peptide.str.len() == 3

results = []
# M1 -- systemic tripeptide, entirely measured
for pepname, ic in IC50_MEASURED_uM.items():
    results.append({"mechanism": "M1 systemic tripeptide", "basis": f"measured ({pepname})",
                    "S": CMAX_HUMAN_uM / ic,
                    "occupancy": CMAX_HUMAN_uM / (CMAX_HUMAN_uM + ic),
                    "model_dependence": "none"})
# M2 -- luminal tripeptide, measured IC50 where available
m2 = pep[tri & pep.measured_IC50_uM.notna()]
s2, o2 = mixture_occupancy(m2.mean_uM, m2.measured_IC50_uM)
results.append({"mechanism": "M2 luminal tripeptide", "basis": f"measured ({len(m2)} peptides)",
                "S": s2, "occupancy": o2, "model_dependence": "none"})
# M3 -- luminal dipeptide, three bases
for tag, col, dep in (("predicted, uncorrected", "pred_IC50_uM", "high"),
                      ("bias-corrected", "corrected_IC50_uM", "moderate"),
                      ("measured subset only", "measured_IC50_uM", "none")):
    d = pep[di].dropna(subset=[col])
    s, o = mixture_occupancy(d.mean_uM, d[col])
    results.append({"mechanism": "M3 luminal dipeptide",
                    "basis": f"{tag} ({len(d)} peptides)", "S": s, "occupancy": o,
                    "model_dependence": dep})
# whole mixture
for tag, col, dep in (("predicted, uncorrected", "pred_IC50_uM", "high"),
                      ("bias-corrected", "corrected_IC50_uM", "moderate")):
    s, o = mixture_occupancy(pep.mean_uM, pep[col])
    results.append({"mechanism": "M2+M3 whole luminal mixture", "basis": f"{tag} (17 peptides)",
                    "S": s, "occupancy": o, "model_dependence": dep})

disc = pd.DataFrame(results)
display(disc.round(4))

naive = float((pep.mean_uM / pep.corrected_IC50_uM).map(lambda x: x / (1 + x)).sum())
correct = float(disc.loc[(disc.mechanism == "M2+M3 whole luminal mixture")
                         & disc.basis.str.startswith("bias"), "occupancy"].iloc[0])
print(f"whole mixture, correct competitive formula : {correct:.3f}")
print(f"  same numbers, occupancies naively summed : {naive:.3f}  <- wrong, shown for contrast")

o_m1 = float(disc.loc[disc.basis == "measured (IPP)", "occupancy"].iloc[0])
print(f"\nM1 systemic (IPP, all measured)            : {o_m1:.3e}")
print(f"M2 luminal tripeptide (all measured)       : {o2:.3f}")
print(f"  ratio, luminal to systemic               : {o2 / o_m1:,.0f}x")
print("\nM1 fails by four orders of magnitude and needs no model to fail -- both its")
print("concentration and its potency are measured. M2 survives on measurement alone.")
print("M3 survives only with a model correction applied, so it is the weakest of the")
print("survivors despite being the most abundant class by concentration.")
pep.to_csv("layer3_peptide_occupancy.csv", index=False)
disc.to_csv("layer3_mechanism_discrimination.csv", index=False)

,mechanism,basis,S,occupancy,model_dependence
0,M1 systemic tripeptide,measured (IPP),0.0002,0.0002,none
1,M1 systemic tripeptide,measured (VPP),0.0001,0.0001,none
2,M2 luminal tripeptide,measured (3 peptides),0.7754,0.4367,none
3,M3 luminal dipeptide,"predicted, uncorrected (13 peptides)",6.2923,0.8629,high
4,M3 luminal dipeptide,bias-corrected (13 peptides),1.0892,0.5213,moderate
5,M3 luminal dipeptide,measured subset only (4 peptides),0.3890,0.2801,none
6,M2+M3 whole luminal mixture,"predicted, uncorrected (17 peptides)",7.0492,0.8758,high
7,M2+M3 whole luminal mixture,bias-corrected (17 peptides),1.8708,0.6517,moderate


whole mixture, correct competitive formula : 0.652
  same numbers, occupancies naively summed : 1.488  <- wrong, shown for contrast

M1 systemic (IPP, all measured)            : 1.794e-04
M2 luminal tripeptide (all measured)       : 0.437
  ratio, luminal to systemic               : 2,435x

M1 fails by four orders of magnitude and needs no model to fail -- both its
concentration and its potency are measured. M2 survives on measurement alone.
M3 survives only with a model correction applied, so it is the weakest of the
survivors despite being the most abundant class by concentration.


---

# 5. Structural context — why Xaa-Pro-Pro binds and is not cleaved

The descriptor model says the C-terminal residues dominate potency. The crystallographic
record says why, and it resolves something Layer 1 raised: VPP and IPP inhibit ACE rather
than being hydrolysed by it, even though ACE is the enzyme they engage.

ACE is a carboxypeptidase with two catalytic domains that differ in their subsites, and
the C-domain is the one responsible for blood-pressure regulation. Of the structures
retrieved here, **4APJ** is the most directly relevant: it holds the bradykinin-potentiating
peptide BPPb as an eleven-residue chain whose **C-terminus is Ile-Pro-Pro** — the same
motif, in the C-domain active site, as an experimental template rather than a docking pose.

The mechanistic point is the proline. A proline at P1 blocks hydrolysis of the preceding
peptide bond, which is why ACE binds Xaa-Pro-Pro tightly without turning it over: the
peptide occupies the site as an inhibitor rather than a substrate. The same chemistry
appears throughout Layer 1 — proline at P1' blocks the X-prolyl dipeptidases, and
aminopeptidases will not hydrolyse the Xaa-Pro imide bond — so one structural feature
explains both why these peptides survive digestion and why they inhibit their own target.

This section reports the structures and the domain differences. It does **not** dock
anything. Docking scores correlate weakly with measured peptide IC50, and the descriptor
model already discriminates the classes this layer needs to separate; adding a docking
number would add precision without adding evidence.

In [ ]:
# ============================================================
# 5. Structural context
# ============================================================
cols = [c for c in ("pdb_id", "resolution_A", "method", "domain_assigned_from_uniprot_range")
        if c in structures.columns]
display(structures[cols])

if "identical" in domains.columns:
    diff = domains[~domains.identical.astype(bool)]
    print(f"active-site positions compared, C-domain vs N-domain : {len(domains)}")
    print(f"  substituted between domains                        : {len(diff)}")
    show = [c for c in ("C_domain_uniprot_pos", "C_domain_residue",
                        "N_domain_uniprot_pos", "N_domain_residue") if c in diff.columns]
    if show:
        display(diff[show].head(15))
    print("\nThese substitutions are why C-domain-selective inhibition is possible at all,")
    print("and why a potency value measured on whole ACE cannot be assigned to one domain.")

if "pdb" in subsites.columns:
    per = subsites.groupby("pdb").size().rename("active_site_residues").reset_index()
    display(per)

,pdb_id,resolution_A,method,domain_assigned_from_uniprot_range
0,1O86,2.00,X-ray,C-domain
1,1O8A,2.00,X-ray,C-domain
2,1UZF,2.00,X-ray,C-domain
3,2OC2,2.25,X-ray,C-domain
4,6H5W,1.37,X-ray,C-domain
5,2C6N,3.00,X-ray,N-domain
6,2C6F,3.01,X-ray,N-domain
7,4CA6,1.91,X-ray,N-domain
8,4APJ,2.60,X-ray,C-domain


active-site positions compared, C-domain vs N-domain : 39
  substituted between domains                        : 15


,C_domain_uniprot_pos,C_domain_residue,N_domain_uniprot_pos,N_domain_residue
0,667,TYR,64,SER
1,671,ASN,68,SER
2,690,ASN,87,ALA
3,726,ASP,126,THR
4,728,GLU,128,GLY
5,729,ARG,129,SER
8,748,GLU,148,SER
9,767,GLU,169,ASP
10,824,SER,226,TYR
12,828,MET,230,TRP



These substitutions are why C-domain-selective inhibition is possible at all,
and why a potency value measured on whole ACE cannot be assigned to one domain.


,pdb,active_site_residues
0,1O86,17
1,1O8A,8
2,1UZF,12
3,2C6N,14
4,2OC2,19
5,4APJ,12
6,4CA6,25
7,6H5W,15


In [ ]:
# ============================================================
# Figure 1 -- the audit and the discrimination, in one view
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.4))

# (a) measured vs out-of-sample predicted, for the peptides the arithmetic uses
lim = [1.0, 1000.0]
ax1.plot(lim, lim, color="0.5", lw=1.0, zorder=1)
ax1.fill_between(lim, [v / 10 for v in lim], [v * 10 for v in lim],
                 color="0.88", lw=0, zorder=0)
for _, r in audit.iterrows():
    colour = "#C2185B" if r.length == 2 else "#00695C"
    ax1.plot(r.measured_IC50_uM, r.loo_pred_IC50_uM, "o", ms=8, mfc=colour,
             mec="white", mew=1.2, zorder=3)
    ax1.annotate(r.peptide, xy=(r.measured_IC50_uM, r.loo_pred_IC50_uM),
                 xytext=(6, -3), textcoords="offset points", fontsize=7.5, color=colour)
ax1.set_xscale("log"); ax1.set_yscale("log"); ax1.set_xlim(lim); ax1.set_ylim(lim)
ax1.set_xlabel("measured IC$_{50}$ (µM)")
ax1.set_ylabel("out-of-sample predicted IC$_{50}$ (µM)")
ax1.set_title("The model over-scores dipeptides (pink)\nand is accurate on tripeptides (green)",
              fontsize=9)
ax1.text(1.6, 420, "10-fold band", fontsize=7, color="0.35")
ax1.text(300, 2.2, "predicted\nmore potent\nthan measured", fontsize=7, color="0.35", ha="center")

# (b) occupancy by mechanism and basis
plot_rows = disc[disc.basis.isin(["measured (IPP)"]) | disc.mechanism.str.startswith(("M2", "M3"))]
plot_rows = plot_rows.sort_values("occupancy")
ypos = np.arange(len(plot_rows))
dep_colour = {"none": "#00695C", "moderate": "#F9A825", "high": "#C2185B"}
ax2.hlines(ypos, 1e-5, plot_rows.occupancy, color="0.80", lw=1.0)
for yv, (_, r) in zip(ypos, plot_rows.iterrows()):
    ax2.plot(r.occupancy, yv, "o", ms=8, mfc=dep_colour[r.model_dependence],
             mec="white", mew=1.2, zorder=3)
ax2.set_yticks(ypos)
ax2.set_yticklabels([f"{r.mechanism.split()[0]}  {r.basis}" for _, r in plot_rows.iterrows()],
                    fontsize=7)
ax2.set_xscale("log"); ax2.set_xlim(1e-5, 2.0)
ax2.set_xlabel("fractional ACE occupancy")
ax2.set_title("Only luminal engagement has arithmetic that closes", fontsize=9)
ax2.axvline(0.1, color="0.6", lw=0.9, ls=":")
ax2.text(0.115, 0.35, "10 % occupancy", fontsize=7, color="0.35", rotation=90,
         va="bottom", ha="left")
for label, key in (("measured only", "none"), ("model, corrected", "moderate"),
                   ("model, uncorrected", "high")):
    ax2.plot([], [], "o", ms=7, mfc=dep_colour[key], mec="white", label=label)
ax2.legend(fontsize=7, frameon=False, loc="lower right")
ax2.spines["left"].set_visible(False); ax2.tick_params(axis="y", length=0)

fig.tight_layout()
fig.savefig("figures/layer3_fig1_audit_and_discrimination.png", dpi=200, bbox_inches="tight")
plt.show()

---

# 6. What would overturn this, and how far it would have to move

A verdict is only useful if it can be reversed by evidence, so this section asks how much
each input would have to change to flip the ranking. Two of the three answers are
reassuring for the conclusion and one is not.

In [ ]:
# ============================================================
# 6. How far would an input have to move?
# ============================================================
tests = []

# (a) how much more potent would VPP/IPP have to be for M1 to reach 10% systemic occupancy?
need_C = 5.0 * 0.10 / (1 - 0.10)
need_ic50 = CMAX_HUMAN_uM * 9                     # IC50 giving 10% occupancy at measured Cmax
best_seq = train.loc[train.pIC50_or_pKi.idxmax(), "sequence"]
best_ic50 = 10 ** (6 - train.pIC50_or_pKi.max())
tests.append({"question": "M1 reaches 10 % systemic occupancy if IPP IC50 were...",
              "answer": f"{need_ic50 * 1e3:.1f} nM",
              "fold_change_required": f"{5.0 / need_ic50:,.0f}x more potent than IPP",
              "plausible": f"not for this motif -- the most potent peptide in the whole "
                           f"training set is {best_seq} at {best_ic50 * 1e3:.0f} nM "
                           f"({best_ic50 / need_ic50:.0f}x weaker still), and every measured "
                           f"Xaa-Pro-Pro is >=5000 nM"})
tests.append({"question": "M1 reaches 10 % if plasma Cmax were...",
              "answer": f"{need_C:.3f} uM",
              "fold_change_required": f"{need_C / CMAX_HUMAN_uM:,.0f}x higher",
              "plausible": "no -- exceeds the ceiling F*D/Vd at every trial dose (Layer 2)"})

# (b) how wrong would the dipeptide bias correction have to be to change M3's verdict?
d = pep[di]
for target in (0.10, 0.50):
    lo, hi = 1e-3, 1e4
    for _ in range(60):
        mid = np.sqrt(lo * hi)
        s = float(np.sum(d.mean_uM / (d.corrected_IC50_uM * mid)))
        if s / (1 + s) > target:
            lo = mid
        else:
            hi = mid
    tests.append({"question": f"M3 dipeptide occupancy would be {target:.0%} if the "
                              f"bias correction were wrong by...",
                  "answer": f"x{np.sqrt(lo * hi):.2f} on IC50",
                  "fold_change_required": f"{np.sqrt(lo * hi):.2f}x",
                  "plausible": "measured spread on this class is ~3-20x, so yes for 10 %"})

# (c) M2 rests on two measured IC50 values -- how sensitive is it to them?
m2c = pep[tri & pep.measured_IC50_uM.notna()]
for mult, tag in ((10.0, "10x weaker"), (0.1, "10x more potent")):
    s, o = mixture_occupancy(m2c.mean_uM, m2c.measured_IC50_uM * mult)
    tests.append({"question": f"M2 luminal tripeptide occupancy if all IC50 were {tag}",
                  "answer": f"{o:.3f}", "fold_change_required": tag,
                  "plausible": "bounds the measurement uncertainty"})
sens = pd.DataFrame(tests)
display(sens)
print("M1 cannot be rescued: both of its inputs are measured and both would have to move")
print("by orders of magnitude, in one case past a physical ceiling. M3 is genuinely")
print("uncertain -- a correction error well within the observed spread moves it across the")
print("10 % line, which is why it is reported as the weakest survivor rather than the")
print("headline. M2 is robust to a 10-fold error in either direction.")
sens.to_csv("layer3_sensitivity.csv", index=False)

,question,answer,fold_change_required,plausible
0,M1 reaches 10 % systemic occupancy if IPP IC50...,8.1 nM,619x more potent than IPP,not for this motif -- the most potent peptide ...
1,M1 reaches 10 % if plasma Cmax were...,0.556 uM,619x higher,no -- exceeds the ceiling F*D/Vd at every tria...
2,M3 dipeptide occupancy would be 10% if the bia...,x9.80 on IC50,9.80x,"measured spread on this class is ~3-20x, so ye..."
3,M3 dipeptide occupancy would be 50% if the bia...,x1.09 on IC50,1.09x,"measured spread on this class is ~3-20x, so ye..."
4,M2 luminal tripeptide occupancy if all IC50 we...,0.072,10x weaker,bounds the measurement uncertainty
5,M2 luminal tripeptide occupancy if all IC50 we...,0.886,10x more potent,bounds the measurement uncertainty


M1 cannot be rescued: both of its inputs are measured and both would have to move
by orders of magnitude, in one case past a physical ceiling. M3 is genuinely
uncertain -- a correction error well within the observed spread moves it across the
10 % line, which is why it is reported as the weakest survivor rather than the
headline. M2 is robust to a 10-fold error in either direction.


---

# 7. Limitations, stated as specifications

Each row names what is wrong, what it costs, and the measurement that would fix it — so
the list reads as a work programme rather than a disclaimer.

In [ ]:
# ============================================================
# 7. Limitations
# ============================================================
limits = pd.DataFrame([
 {"#": 1,
  "limitation": "Intestinal/brush-border ACE inhibition is not established to lower blood pressure",
  "consequence": "M2 and M3 are arithmetically possible mechanisms, NOT demonstrated ones; this is the single largest gap in the whole three-layer argument",
  "measurement that would resolve it": "a blood-pressure endpoint under a non-absorbed ACE inhibitor, or under luminal ACE knockout/inhibition in an animal model",
  "priority": "highest"},
 {"#": 2,
  "limitation": "Only 7.9 % of retrieved ChEMBL ACE records resolved to a peptide sequence",
  "consequence": "73 training peptides, 52 of them dipeptides; longer peptides are extrapolation and the set cannot support a high-capacity model",
  "measurement that would resolve it": "sequence-annotated peptide IC50 sets (AHTPDB, BIOPEP-UWM) -- neither is reachable from this environment's network allowlist",
  "priority": "high"},
 {"#": 3,
  "limitation": "Potency for 10 of 17 luminal peptides is model-predicted, not measured",
  "consequence": "M3 depends on a bias correction estimated from only 5 dipeptides; Section 6 shows an error within the observed spread moves its verdict",
  "measurement that would resolve it": "ACE IC50 for FP, EP, YP, LP, MP, HP, QP against the same enzyme preparation and substrate",
  "priority": "highest (for the next hypothesis)"},
 {"#": 4,
  "limitation": "Layer 2 indicates absorption may be gastric, not jejunal",
  "consequence": "Layer 1's peptidome may describe the wrong compartment, so the CONCENTRATIONS entering M2 and M3 could be wrong even if the potencies are right",
  "measurement that would resolve it": "gastric and duodenal aspirate peptide profiling after a defined fermented-milk intake",
  "priority": "highest"},
 {"#": 5,
  "limitation": "IC50 values are pooled across assays, substrates and enzyme sources",
  "consequence": "a portion of the training spread is methodological rather than biological, inflating apparent prediction error and possibly biasing it",
  "measurement that would resolve it": "a single-laboratory panel measured on one substrate, reported as Ki",
  "priority": "high"},
 {"#": 6,
  "limitation": "Free luminal ACE concentration and the free fraction of each peptide are unknown",
  "consequence": "occupancy is computed as though all peptide is free and available; binding to mucus, food matrix or other proteases would reduce it",
  "measurement that would resolve it": "free peptide fraction in intestinal aspirate, and brush-border ACE quantification",
  "priority": "medium"},
 {"#": 7,
  "limitation": "Potency is measured on whole ACE, not per catalytic domain",
  "consequence": "C-domain selectivity cannot be assigned, yet only the C-domain is relevant to blood pressure (Section 5)",
  "measurement that would resolve it": "IC50 against isolated C- and N-domain constructs",
  "priority": "medium"},
 {"#": 8,
  "limitation": "No docking or free-energy calculation was performed",
  "consequence": "the structural argument in Section 5 is interpretive; no per-peptide binding mode is computed",
  "measurement that would resolve it": "not a measurement -- a compute decision, deferred because docking scores correlate weakly with peptide IC50",
  "priority": "low"},
])
display(limits)
print("Items 1 and 4 are the two that could invalidate the surviving mechanisms outright.")
print("Item 3 is the highest-value new measurement, because it is what would settle")
print("whether the dipeptides matter -- seven IC50 determinations on one plate.")
limits.to_csv("layer3_limitations.csv", index=False)

,#,limitation,consequence,measurement that would resolve it,priority
0,1,Intestinal/brush-border ACE inhibition is not ...,M2 and M3 are arithmetically possible mechanis...,a blood-pressure endpoint under a non-absorbed...,highest
1,2,Only 7.9 % of retrieved ChEMBL ACE records res...,"73 training peptides, 52 of them dipeptides; l...","sequence-annotated peptide IC50 sets (AHTPDB, ...",high
2,3,Potency for 10 of 17 luminal peptides is model...,M3 depends on a bias correction estimated from...,"ACE IC50 for FP, EP, YP, LP, MP, HP, QP agains...",highest (for the next hypothesis)
3,4,"Layer 2 indicates absorption may be gastric, n...",Layer 1's peptidome may describe the wrong com...,gastric and duodenal aspirate peptide profilin...,highest
4,5,"IC50 values are pooled across assays, substrat...",a portion of the training spread is methodolog...,a single-laboratory panel measured on one subs...,high
5,6,Free luminal ACE concentration and the free fr...,occupancy is computed as though all peptide is...,"free peptide fraction in intestinal aspirate, ...",medium
6,7,"Potency is measured on whole ACE, not per cata...","C-domain selectivity cannot be assigned, yet o...",IC50 against isolated C- and N-domain constructs,medium
7,8,No docking or free-energy calculation was perf...,the structural argument in Section 5 is interp...,"not a measurement -- a compute decision, defer...",low


Items 1 and 4 are the two that could invalidate the surviving mechanisms outright.
Item 3 is the highest-value new measurement, because it is what would settle
whether the dipeptides matter -- seven IC50 determinations on one plate.


---

# 8. What the three layers together support

**Systemic ACE inhibition by intact VPP/IPP is not the mechanism.** This is the firmest
conclusion in the project and the one that needs the least modelling: measured plasma
Cmax and measured IC50 give an occupancy near 10⁻⁴, and Section 6 shows neither input
can move far enough to rescue it. Layer 2 independently reached the same place from
absorption kinetics.

**Luminal engagement is the only candidate whose arithmetic closes.** In the gut lumen,
Layer 1's concentrations and measured IC50 values land in the same range for the first
time anywhere in this project. On measurement alone, the two lactotripeptides reach
substantial occupancy of luminal ACE — no model involved. That is a genuine finding and a
genuine surprise, given that the tripeptides are a minor molar component of the mixture.

**The Xaa-Pro dipeptides are not the actors the earlier draft suggested.** They dominate
the lumen by concentration, roughly eighty-fold over the tripeptides, but the two most
abundant are measured at several hundred micromolar and are simply weak inhibitors.
Abundance did not compensate. Their contribution is real but secondary, and it rests on a
bias correction rather than on measurement.

**The decisive unknown is not a potency — it is a location and a link.** Two things would
settle far more than another model: where absorption and luminal exposure actually occur
(Layer 2's tmax anomaly, Layer 3 item 4), and whether inhibiting ACE in the gut affects
blood pressure at all (item 1). Neither is a modelling question. Both are experiments.

## The falsification this project can stand behind

A digital twin was built to predict how well anti-hypertensive peptides work. What it
produced instead is a quantitative demonstration that the accepted mechanism cannot
account for the observed clinical effect, plus a ranked shortlist of what might. The
framework did its job — it just did not return the answer it was designed to confirm,
which is the outcome that carries information.

In [ ]:
# ============================================================
# Exports
# ============================================================
import glob
for f in sorted(glob.glob("layer3_*.csv")) + sorted(glob.glob("figures/layer3_*.png")):
    print(f"  {f}  ({os.path.getsize(f) / 1024:.1f} KB)")
print("\nThese plus the stdout log are the complete Layer 3 handback.")

  layer3_limitations.csv  (2.4 KB)
  layer3_mechanism_discrimination.csv  (0.8 KB)
  layer3_peptide_occupancy.csv  (2.7 KB)
  layer3_sensitivity.csv  (0.9 KB)
  figures/layer3_fig1_audit_and_discrimination.png  (171.0 KB)

These plus the stdout log are the complete Layer 3 handback.
